# 5. From recognized signs to subtitles

**Sign recognition** (notebooks 3/4) answers: "which single sign is this?"

**Sign language translation** is a different, harder problem: converting a
full ISL sentence (with its own grammar) into fluent English. This project
does the simpler thing: take recognized words and format them into readable
English — NOT real grammatical translation. That's a stated limitation,
not something to overclaim in a report.

## 0. Setup (Colab or local)

In [ ]:
import os

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/ISL_Sign2Sub"
    ON_COLAB = True
except ImportError:
    PROJECT_DIR = os.path.abspath("..")
    ON_COLAB = False

OUTPUT_DIR = os.path.join(PROJECT_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Running on Colab:", ON_COLAB)
print("Subtitles will be saved to:", OUTPUT_DIR)

## 1. Words to sentence

Our sign names use underscores (e.g. `thank_you`), so they're cleaned up
before formatting.

In [ ]:
def words_to_sentence(words):
    """Simple word-to-sentence formatter (capitalization + punctuation).
    This is NOT grammar translation.
    """
    if not words:
        return ""
    sentence = " ".join(w.capitalize() for w in words)
    question_words = {"how", "what", "why", "where", "who"}
    if words[0].lower() in question_words:
        sentence += "?"
    else:
        sentence += "."
    return sentence

def clean_word(w):
    return w.replace("_", " ").lower()

## 2. SRT and TXT generation

In [ ]:
def format_srt_timestamp(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds - int(seconds)) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"

def generate_srt(subtitle_lines, seconds_per_line=2.0, out_path=None):
    out_path = out_path or os.path.join(OUTPUT_DIR, "subtitles.srt")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as f:
        for i, line in enumerate(subtitle_lines):
            start = i * seconds_per_line
            end = start + seconds_per_line
            f.write(f"{i+1}\n")
            f.write(f"{format_srt_timestamp(start)} --> {format_srt_timestamp(end)}\n")
            f.write(f"{line}\n\n")
    return out_path

def generate_txt(sentence, out_path=None):
    out_path = out_path or os.path.join(OUTPUT_DIR, "subtitles.txt")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as f:
        f.write(sentence)
    return out_path

print("Functions ready.")

## 3. Generate real subtitles from real predictions

Replace `predicted_words` below with your own actual predictions from
`4_test.ipynb` (run `predict_video()` on a few different sign videos one
after another to simulate a short phrase).

In [ ]:
predicted_words = ["hello", "thank_you", "alright", "good_morning"]   # replace with your real predictions
cleaned = [clean_word(w) for w in predicted_words]

sentence = words_to_sentence(cleaned)
print("Sentence:", sentence)

srt_lines = [w.capitalize() for w in cleaned]
srt_path = generate_srt(srt_lines)
txt_path = generate_txt(sentence)

print("Saved:", srt_path)
print("Saved:", txt_path)

Open `output/subtitles.srt` (in Drive, or locally) to confirm it shows
numbered blocks with timestamps and your real words, e.g.:
```text
1
00:00:00,000 --> 00:00:02,000
Hello

2
00:00:02,000 --> 00:00:04,000
Thank you
...
```